# 18｜Transformer_注意力架構與預訓練

位置編碼加上 causal mask，驗證第 t 個 token 不會看到未來。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
n, d = 6, 4
pos = np.arange(n)[:, None]
freq = np.exp(np.arange(0, d, 2) * (-np.log(10000.0) / d))
pe = np.zeros((n, d)); pe[:, 0::2] = np.sin(pos * freq); pe[:, 1::2] = np.cos(pos * freq)
scores = pe @ pe.T / np.sqrt(d)
causal_mask = np.triu(np.ones((n, n), dtype=bool), k=1)
masked = np.where(causal_mask, -1e9, scores)
weights = np.exp(masked - masked.max(axis=1, keepdims=True)); weights /= weights.sum(1, keepdims=True)
print('future attention mass:', weights[causal_mask].sum())
pd.DataFrame(weights).round(3)

## 執行結果圖

In [ ]:
plt.imshow(weights, cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='attention weight'); plt.xlabel('key position'); plt.ylabel('query position')
plt.title('Causal self-attention is lower triangular')
savefig('18_transformer_mask_demo')
report('transformer_mask', {'future_mass': weights[causal_mask].sum(), 'weights': weights.tolist()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。